# News filter bot — notebook (prod)

Ручной запуск дайджеста новостей.

- Источники: `sources.json`
- Секреты: `config_local.json` (токен и chat_id только там)
- Дедуп: `seen_news.json`
- Лог: `outputs/news_log.csv`

Перед запуском: закрой `news_log.csv`, если он открыт в Excel.


## 1) Установка зависимостей (один раз)
Если уже установлено — можно пропустить.


In [ ]:
!pip -q install requests beautifulsoup4 dateparser

## 2) Загрузка config и sources


In [ ]:
import json
from pathlib import Path

def load_json(path: Path, default):
    try:
        if not path.exists():
            return default
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def save_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# 1) config
CONFIG_PATH = Path(r"C:\Users\iribs\Documents\news_filter_bot\config_local.json")
cfg = load_json(CONFIG_PATH, default={})

# 2) paths из config
if "files" not in cfg:
    raise ValueError("В config_local.json нет секции 'files'. Проверь cfg['files']['sources_file'], cfg['files']['seen_file'], cfg['files']['log_file'].")

sources_path = Path(cfg["files"]["sources_file"])
seen_path = Path(cfg["files"]["seen_file"])
log_path = Path(cfg["files"]["log_file"])

# 3) sources
sources = load_json(sources_path, default=[])
if not isinstance(sources, list):
    raise ValueError("sources.json должен быть списком (list) объектов-источников.")

# 4) добавляем/обновляем источники
NEW_SOURCES = [
    {"id":"gazpromneft_ir","type":"html","url":"https://ir.gazprom-neft.ru/news-and-events/news/","parser":"gazpromneft_ir","enabled":True,"priority":80,"tags":["SIBN","Газпромнефть","IR","news"]},
    {"id":"eurotrans_news","type":"html","url":"https://www.evrotrans-ao.ru/news/","parser":"eurotrans_news","enabled":True,"priority":70,"tags":["EUTR","Евротранс","news","corp"]},
    {"id":"mtsbank_news","type":"html","url":"https://www.mtsbank.ru/o-banke/novosti/2026/","parser":"mtsbank_news","enabled":True,"priority":80,"tags":["MBNK","МТС Банк","press","news"]},
    {"id":"rossetimr_press","type":"html","url":"https://rossetimr.ru/en/about/press/company_news/","parser":"rossetimr_press","enabled":True,"priority":80,"tags":["MSRS","Россети Московский регион","press","news"]},
    {"id":"novatek_press","type":"html","url":"https://www.novatek.ru/ru/press/","parser":"novatek_press","enabled":True,"priority":80,"tags":["NVTK","Новатэк","press","corp"]},
    {"id":"rostelecom_press","type":"html","url":"https://www.company.rt.ru/press/news/","parser":"rostelecom_press","enabled":True,"priority":70,"tags":["RTKM","Ростелеком","press","news"]},
    {"id":"surgutneftegas_investors","type":"html","url":"https://www.surgutneftegas.ru/investors/","parser":"surgutneftegas_investors","enabled":True,"priority":60,"tags":["SNGS","Сургутнефтегаз","investors","corp"]},
    {"id":"x5_press","type":"html","url":"https://www.x5.ru/en/press-center/press-releases/","parser":"x5_press","enabled":True,"priority":80,"tags":["X5","FIVE","press","corp"]},
    {"id":"smartlab_x5","type":"html","url":"https://smart-lab.ru/company/x5retailgroup/blog","parser":"smartlab_company_blog","enabled":True,"priority":80,"tags":["X5","FIVE","smart-lab","corp"]},
]

existing_by_id = {s.get("id"): s for s in sources if isinstance(s, dict) and s.get("id")}
added, updated = 0, 0

for ns in NEW_SOURCES:
    sid = ns["id"]
    if sid not in existing_by_id:
        sources.append(ns)
        added += 1
    else:
        cur = existing_by_id[sid]
        changed = False
        for k in ["type", "url", "parser", "enabled", "priority", "tags"]:
            if ns.get(k) is not None and cur.get(k) != ns.get(k):
                cur[k] = ns[k]
                changed = True
        if changed:
            updated += 1

save_json(sources_path, sources)

# 5) enabled_sources для следующих ячеек
enabled_sources = [s for s in sources if s.get("enabled", True)]

print("[OK] config:", CONFIG_PATH)
print("[OK] sources:", sources_path)
print(f"[OK] sources merged: added={added}, updated={updated}, total={len(sources)}, enabled={len(enabled_sources)}")
print("[OK] seen file:", seen_path)
print("[OK] log file:", log_path)



## 3) Дедуп + утилиты


In [ ]:
import hashlib
from datetime import datetime, timezone, timedelta
from pathlib import Path

def make_uid(url: str, title: str = "") -> str:
    raw = ((url or "").strip().lower() + "|" + (title or "").strip().lower()).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def load_seen(path: str) -> dict:
    p = Path(path)
    if not p.exists():
        return {}
    try:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}

def save_seen(path: str, seen: dict) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(seen, f, ensure_ascii=False, indent=2)

def prune_seen(seen: dict, keep_days: int = 180) -> dict:
    cutoff = datetime.now(timezone.utc) - timedelta(days=keep_days)
    out = {}
    for uid, iso in seen.items():
        try:
            dt = datetime.fromisoformat(iso.replace("Z", "+00:00"))
            if dt >= cutoff:
                out[uid] = iso
        except Exception:
            pass
    return out

def contains_any_keyword(title: str, keywords: list[str]) -> bool:
    t = (title or "").lower()
    return any(k.lower() in t for k in keywords)

def chunk_text_blocks(blocks: list[str], max_len: int = 3500) -> list[str]:
    parts = []
    cur = ""
    for b in blocks:
        b = (b or "").strip()
        if not b:
            continue
        add = (("\n\n" if cur else "") + b)
        if len(cur) + len(add) <= max_len:
            cur += add
        else:
            if cur:
                parts.append(cur)
            cur = b
    if cur:
        parts.append(cur)
    return parts

## 4) Telegram отправка (без превью)


In [ ]:
import requests

def tg_send(cfg: dict, text: str) -> dict:
    token = cfg["telegram"]["bot_token"].strip()
    chat_id = cfg["telegram"]["chat_id"]
    disable_preview = bool(cfg["telegram"].get("disable_preview", True))

    url = f"https://api.telegram.org/bot{token}/sendMessage"
    payload = {
        "chat_id": chat_id,
        "text": text,
        "disable_web_page_preview": disable_preview,
    }
    r = requests.post(url, json=payload, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Telegram HTTP {r.status_code}: {r.text[:300]}")
    return r.json()

## 5) Парсер Smart-Lab (company blog)
Дата берётся из `li.date` на странице поста и переводится в ISO (Мск).


In [ ]:
import re
import dateparser
from bs4 import BeautifulSoup

TZ_MOSCOW = "Europe/Moscow"

def fetch_post_published_dt(post_url: str):
    try:
        r = requests.get(post_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        li = soup.find("li", class_="date")
        if li:
            raw = li.get_text(" ", strip=True)
            settings = {
                "RETURN_AS_TIMEZONE_AWARE": True,
                "TIMEZONE": TZ_MOSCOW,
                "TO_TIMEZONE": TZ_MOSCOW,
                "DATE_ORDER": "DMY",
            }
            return dateparser.parse(raw, languages=["ru"], settings=settings)
    except Exception:
        return None
    return None

def parse_smartlab_company_blog(source: dict, max_items: int = 30) -> list[dict]:
    url = source["url"]
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    good = []
    seen = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not re.search(r"^/company/[^/]+/blog/\d+\.php$", href):
            continue

        title = (a.get_text(" ", strip=True) or "").strip()
        if not title:
            continue

        low = title.lower()
        if low in ("читать дальше", "комментировать") or low.startswith("комментар"):
            continue

        if href in seen:
            continue
        seen.add(href)
        good.append((href, title))

        if len(good) >= max_items:
            break

    items = []
    base = "https://smart-lab.ru"
    for href, title in good:
        post_url = base + href
        dt = fetch_post_published_dt(post_url)
        items.append({
            "source_name": source["name"],
            "title": title,
            "url": post_url,
            "published_dt": dt,
            "published_iso": dt.isoformat() if dt else ""
        })
    return items

## 6) CSV лог
Если файл занят (например открыт в Excel), будет понятная ошибка.


In [ ]:
import csv

def write_csv_log(csv_path: str, rows: list[dict]):
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = path.exists()

    fieldnames = ["fetched_at_iso", "source_name", "bucket", "published_iso", "title", "url"]

    try:
        with open(path, "a", encoding="utf-8-sig", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                w.writeheader()
            for r in rows:
                w.writerow(r)
    except PermissionError:
        raise PermissionError(
            f"Файл лога занят (скорее всего открыт в Excel): {csv_path}. Закрой файл и запусти снова."
        )

## 7) Универсальный запуск дайджеста
- Собирает все включённые источники
- Фильтрует по N дней (по дате публикации)
- Дедуп по `seen_news.json`
- Делит на важное/остальное
- Отправляет 1–3 сообщения в Telegram
- Записывает в `outputs/news_log.csv`


In [ ]:
# 7) Универсальный запуск дайджеста (мульти-источники + дедуп + лог)

import re
import csv
from datetime import datetime, timedelta, timezone
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

UTC = timezone.utc

def now_utc():
    return datetime.now(tz=UTC)

def clean_text(s: str) -> str:
    if not s:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def make_uid(item):
    # важно: uid должен быть стабилен; используем url+title
    return (item.get("url") or "") + "|" + (item.get("title") or "")

def to_dt(value):
    if isinstance(value, datetime):
        return value.astimezone(UTC)
    if not value:
        return None

    s = clean_text(str(value))

    # dd.mm.yyyy
    m = re.search(r"(\d{1,2})\.(\d{1,2})\.(\d{4})", s)
    if m:
        d, mo, y = map(int, m.groups())
        return datetime(y, mo, d, tzinfo=UTC)

    # RU: "23 января 2026"
    ru_mon = {
        "января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6,
        "июля": 7, "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12
    }
    m = re.search(r"(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+(\d{4})", s, re.I)
    if m:
        day = int(m.group(1))
        mon = m.group(2).lower()
        year = int(m.group(3))
        return datetime(year, ru_mon[mon], day, tzinfo=UTC)

    # EN: "3 February 2026"
    m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", s)
    if m:
        day = int(m.group(1))
        mon = m.group(2).lower()
        year = int(m.group(3))
        mon_map = {
            "january": 1, "february": 2, "march": 3, "april": 4,
            "may": 5, "june": 6, "july": 7, "august": 8,
            "september": 9, "october": 10, "november": 11, "december": 12
        }
        if mon in mon_map:
            return datetime(year, mon_map[mon], day, tzinfo=UTC)

    # EN short: "13 Jan 2026"
    m = re.search(r"(\d{1,2})\s+([A-Za-z]{3})\s+(\d{4})", s)
    if m:
        day = int(m.group(1))
        mon3 = m.group(2).lower()
        year = int(m.group(3))
        mon_map3 = {
            "jan": 1, "feb": 2, "mar": 3, "apr": 4,
            "may": 5, "jun": 6, "jul": 7, "aug": 8,
            "sep": 9, "oct": 10, "nov": 11, "dec": 12
        }
        if mon3 in mon_map3:
            return datetime(year, mon_map3[mon3], day, tzinfo=UTC)

    return None

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "news_filter_bot/1.0 (requests)"
})

def get_soup(url, timeout=40, retries=1):
    last = None
    for _ in range(retries + 1):
        try:
            r = SESSION.get(url, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except Exception as e:
            last = e
    raise last

def try_extract_date_near(a_tag, patterns):
    parent = a_tag.parent
    for _ in range(6):
        if not parent:
            break
        txt = clean_text(parent.get_text(" "))
        for pat in patterns:
            m = re.search(pat, txt, re.I)
            if m:
                return m.group(0)
        parent = parent.parent
    return ""

# ----------------------------
# Smart-Lab parser
# ----------------------------
def parser_smartlab_company_blog(source):
    url = source["url"]
    soup = get_soup(url)
    slug = url.rstrip("/").split("/")[-2]

    items = []
    date_patterns = (
        r"\d{1,2}\.\d{1,2}\.\d{4}",
        r"\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}",
    )

    for a in soup.select(f'a[href*="/company/{slug}/blog/"]'):
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if (not href) or (not title):
            continue

        full = href if href.startswith("http") else urljoin("https://smart-lab.ru", href)

        # --- Smart-Lab cleanup: оставляем только реальные посты ---

        # убираем mobile
        if "/mobile/" in full:
            continue

        # убираем пагинацию /blog/pageN/
        if re.search(r"/blog/page\d+/?$", full):
            continue

        # убираем комментарии
        if "#comments" in full:
            continue

        # оставляем только реальные посты вида .../blog/1234567.php
        if not re.search(r"/blog/\d+\.php$", full):
            continue

        # фильтр по мусорным заголовкам
        t = title.strip().lower()

        if t in {"читать дальше", "новый дизайн", "сюда →", "← туда"}:
            continue

        if re.fullmatch(r"\d+", title.strip()):
            continue

        if len(title.strip()) < 10:
            continue

        date_text = try_extract_date_near(a, date_patterns)

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": to_dt(date_text),
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

# ----------------------------
# parsers (MVP)
# ----------------------------
def parse_x5_pressreleases(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for a in soup.select('a[href*="/en/news/"], a[href*="/ru/news/"]'):
        title = clean_text(a.get_text(" "))
        href = a.get("href")
        if not href or not title:
            continue
        full = href if href.startswith("http") else urljoin(url, href)

        date_text = try_extract_date_near(a, [r"\d{1,2}\s+[A-Za-z]+\s+\d{4}"])
        dt = to_dt(date_text)

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

def _approx_dt_from_title(title: str):
    t = title.lower()

    m = re.search(r"(20\d{2})", t)
    year = int(m.group(1)) if m else None
    if not year:
        return None

    if ("9 месяц" in t) or ("9 мес" in t) or ("9m" in t):
        return datetime(year, 10, 1, tzinfo=UTC)
    if ("6 месяц" in t) or ("6 мес" in t) or ("6m" in t) or ("полугод" in t) or ("1 полугод" in t):
        return datetime(year, 7, 1, tzinfo=UTC)
    if ("3 месяц" in t) or ("3 мес" in t) or ("3m" in t) or ("квартал" in t) or ("1 кв" in t) or ("i кв" in t):
        return datetime(year, 4, 1, tzinfo=UTC)
    if ("12 месяц" in t) or ("12 мес" in t) or ("год" in t) or ("annual" in t):
        return datetime(year + 1, 3, 1, tzinfo=UTC)

    return datetime(year, 6, 30, tzinfo=UTC)

def parse_gazpromneft_ir_financial_results(source):
    url = source["url"]
    soup = get_soup(url, timeout=25, retries=0)

    items = []

    date_patterns = [
        r"\d{1,2}\.\d{1,2}\.\d{4}",
        r"\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}",
    ]

    for a in soup.select('a[href*=".pdf"], a[href$=".pdf"]'):
        href = a.get("href")
        if not href:
            continue

        full = href if href.startswith("http") else urljoin(url, href)

        title = clean_text(a.get_text(" "))
        if not title:
            title = full.split("/")[-1]

        date_text = try_extract_date_near(a, date_patterns)
        dt = to_dt(date_text)

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())



def parse_rossetimr_company_news(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for a in soup.select('a[href*="item"]'):
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if not href or not title:
            continue
        full = href if href.startswith("http") else urljoin(url, href)

        date_text = try_extract_date_near(a, [r"\d{1,2}\.\d{1,2}\.\d{4}"])
        dt = to_dt(date_text)

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

def parse_mtsbank_news_2026(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for a in soup.select('a[href*="/o-banke/novosti/detail/"]'):
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if not href or not title:
            continue

        full = href if href.startswith("http") else urljoin(url, href)

        dt = None
        try:
            detail = get_soup(full)
            page_txt = clean_text(detail.get_text(" "))
            m = re.search(r"от\s+(\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4})", page_txt, re.I)
            if m:
                dt = to_dt(m.group(1))
            else:
                m = re.search(r"(\d{1,2}\s+[A-Za-z]{3}\s+\d{4})", page_txt)
                if m:
                    dt = to_dt(m.group(1))
        except Exception:
            dt = None

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

def parse_eurotrans_news(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for art in soup.select("article"):
        a = art.select_one("a[href]")
        if not a:
            continue
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if not href or not title:
            continue
        full = href if href.startswith("http") else urljoin(url, href)

        dt = None
        t = art.select_one("time")
        if t:
            dt = to_dt(t.get("datetime") or t.get_text(" "))
        else:
            date_text = clean_text(art.get_text(" "))
            m = re.search(r"(\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4})", date_text, re.I)
            if m:
                dt = to_dt(m.group(1))

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    if not items:
        for a in soup.select('a[href^="https://www.evrotrans-ao.ru/"]'):
            href = a.get("href")
            title = clean_text(a.get_text(" "))
            if not href or not title:
                continue
            if len(title) < 12:
                continue
            if href.rstrip("/").endswith("/news"):
                continue

            items.append({
                "source_id": source.get("id"),
                "source_name": source.get("name") or source.get("id"),
                "title": title,
                "url": href,
                "published": None,
            })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

def parse_novatek_press(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for a in soup.select('a[href*="releases/index.php?id_4="]'):
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if not href or not title:
            continue
        full = href if href.startswith("http") else urljoin(url, href)

        date_text = try_extract_date_near(a, [
            r"\d{1,2}\.\d{1,2}\.\d{4}",
            r"\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}",
        ])
        dt = to_dt(date_text)

        if dt is None:
            try:
                detail = get_soup(full)
                txt = clean_text(detail.get_text(" "))
                m = re.search(r"(\d{1,2}\.\d{1,2}\.\d{4})", txt)
                if m:
                    dt = to_dt(m.group(1))
                else:
                    m = re.search(r"(\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4})", txt, re.I)
                    if m:
                        dt = to_dt(m.group(1))
            except Exception:
                dt = None

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

def parse_rostelecom_press_news(source):
    url = source["url"]
    soup = get_soup(url)
    items = []

    for a in soup.select('a[href*="/press/news/"]'):
        href = a.get("href")
        title = clean_text(a.get_text(" "))
        if not href or not title:
            continue
        if len(title) < 12:
            continue

        full = href if href.startswith("http") else urljoin(url, href)

        date_text = try_extract_date_near(a, [
            r"\d{1,2}\.\d{1,2}\.\d{4}",
            r"\d{1,2}\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+\d{4}",
        ])
        dt = to_dt(date_text)

        items.append({
            "source_id": source.get("id"),
            "source_name": source.get("name") or source.get("id"),
            "title": title,
            "url": full,
            "published": dt,
        })

    uniq = {}
    for it in items:
        uniq[it["url"]] = it
    return list(uniq.values())

# ----------------------------
# registry
# ----------------------------
PARSERS = {
    "x5_press": parse_x5_pressreleases,
    "rossetimr_press": parse_rossetimr_company_news,
    "mtsbank_news": parse_mtsbank_news_2026,
    "eurotrans_news": parse_eurotrans_news,
    "novatek_press": parse_novatek_press,
    "rostelecom_press": parse_rostelecom_press_news,
    "gazpromneft_ir_financial_results": parse_gazpromneft_ir_financial_results,
}


# ----------------------------
# run all enabled sources
# ----------------------------
def collect_all_sources(enabled_sources):
    collected = []
    errors = []

    for s in enabled_sources:
        parser_name = s.get("parser")
        if not parser_name:
            continue

        if parser_name == "smartlab_company_blog":
            try:
                collected.extend(parser_smartlab_company_blog(s))
            except Exception as e:
                errors.append((s.get("id"), parser_name, str(e)))
            continue

        fn = PARSERS.get(parser_name)
        if not fn:
            continue

        try:
            collected.extend(fn(s))
        except Exception as e:
            errors.append((s.get("id"), parser_name, str(e)))

    return collected, errors

# ----------------------------
# params/files (from cfg loaded in cell 2)
# ----------------------------
N_DAYS = int(cfg.get("params", {}).get("days", 7))
LIMIT_TOTAL = int(cfg.get("params", {}).get("limit", 10))
IMPORTANT_TOP = int(cfg.get("params", {}).get("important", 5))

SEEN_PATH = Path(cfg["files"]["seen_file"])
LOG_PATH = Path(cfg["files"]["log_file"])

seen = load_json(SEEN_PATH, default={"seen": []})
seen_set = set(seen.get("seen", []))

cutoff = now_utc() - timedelta(days=N_DAYS)

items, errors = collect_all_sources(enabled_sources)

# нормализуем даты
for it in items:
    it["published"] = to_dt(it.get("published")) or datetime(1900, 1, 1, tzinfo=UTC)

# фильтр по времени
items = [it for it in items if it["published"] >= cutoff]

# ВАЖНО: гарантируем uid для ВСЕХ items (нужно для fallback)
for it in items:
    it["uid"] = make_uid(it)

# дедуп по seen_news.json
fresh = []
for it in items:
    if it["uid"] in seen_set:
        continue
    fresh.append(it)

# fallback: если нового нет, показываем просто последние N (без дедупа)
if not fresh:
    print("[info] no new items after dedup -> fallback to latest items")
    fresh = items[:]  # items уже отфильтрованы по датам и уже имеют uid

# сортировка: новые сверху
fresh.sort(key=lambda x: x["published"], reverse=True)

# лимит
fresh = fresh[:LIMIT_TOTAL]

# обновим seen
for it in fresh:
    seen_set.add(it["uid"])
seen["seen"] = list(seen_set)
save_json(SEEN_PATH, seen)

# ----------------------------
# log to CSV
# ----------------------------
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
is_new = not LOG_PATH.exists()

with open(LOG_PATH, "a", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    if is_new:
        w.writerow(["ts_utc", "source_id", "source_name", "published_utc", "title", "url"])
    ts = now_utc().isoformat()
    for it in fresh:
        w.writerow([
            ts,
            it.get("source_id", ""),
            it.get("source_name", ""),
            it.get("published").isoformat() if it.get("published") else "",
            it.get("title", ""),
            it.get("url", ""),
        ])

# ----------------------------
# build digest text (универсальный заголовок из source_name)
# ----------------------------
def fmt_dt(dt):
    return dt.strftime("%Y-%m-%d") if dt else ""

lines = []
lines.append(f"Дайджест за {N_DAYS} дней. Лимит: {LIMIT_TOTAL}. Важное: {IMPORTANT_TOP}.")
lines.append("")

for i, it in enumerate(fresh, start=1):
    mark = "(!) " if i <= IMPORTANT_TOP else ""
    src = it.get("source_name") or it.get("source_id") or "source"
    dt = fmt_dt(it.get("published"))
    title = it.get("title", "")
    url = it.get("url", "")
    lines.append(f"{i}. {mark}{src} | {dt} | {title}")
    lines.append(f"   {url}")

digest_text = "\n".join(lines)
print(digest_text)

if errors:
    print(f"[debug] collected_total={len(items)}")
    print(f"[debug] fresh_after_seen={len(fresh)}")
    print("\n[WARN] parser errors:")
    for sid, p, msg in errors[:20]:
        print(f"- {sid} / {p}: {msg}")


Ячейка 8. Отправка

In [ ]:
# 8) Send digest to Telegram (short + reliable: chunking + message_id + clear errors)

import requests

TG_TOKEN = cfg["telegram"]["bot_token"]
CHAT_ID = cfg["telegram"]["chat_id"]
BASE = f"https://api.telegram.org/bot{TG_TOKEN}"

def tg_call(method, payload=None, timeout=25):
    r = requests.post(f"{BASE}/{method}", json=(payload or {}), timeout=timeout)
    try:
        data = r.json()
    except Exception:
        print("[tg] non-json:", r.status_code, r.text[:500])
        r.raise_for_status()
        return None
    if not data.get("ok", False):
        # здесь всегда будет причина, если Telegram не принял сообщение
        raise RuntimeError(f"Telegram API error: {data}")
    return data

def split_telegram(text, limit=3900):
    parts, cur, cur_len = [], [], 0
    for line in str(text).split("\n"):
        add = line + "\n"
        if cur_len + len(add) > limit and cur:
            parts.append("".join(cur).rstrip())
            cur, cur_len = [add], len(add)
        else:
            cur.append(add)
            cur_len += len(add)
    if cur:
        parts.append("".join(cur).rstrip())
    return parts or [str(text)[:limit]]

# digest_text формируется в ячейке 7
if "digest_text" not in globals() or not str(digest_text).strip():
    raise RuntimeError("digest_text is empty or not defined. Run cell 7 first.")

parts = split_telegram(digest_text)

sent_ids = []
for p in parts:
    resp = tg_call("sendMessage", {
        "chat_id": CHAT_ID,
        "text": p,
        "disable_web_page_preview": True
    })
    mid = resp["result"]["message_id"]
    sent_ids.append(mid)
    print(f"[tg] sent message_id={mid}, chars={len(p)}")

# мини-проверка: бот отвечает на последнее сообщение (если клиент глючит — это спасает)
last_id = sent_ids[-1]
resp2 = tg_call("sendMessage", {
    "chat_id": CHAT_ID,
    "text": f"verify: previous digest message_id={last_id}",
    "reply_to_message_id": last_id,
    "disable_web_page_preview": True
})
print(f"[tg] verify reply message_id={resp2['result']['message_id']} -> replied_to={last_id}")

print(f"[telegram] sent {len(sent_ids)} message(s) to chat_id={CHAT_ID}. ids={sent_ids}")
